# NIH ChestX-ray14 — DenseNet-121, split por paciente, Grad-CAM y sesgo

> ⚠️ **NO ES UNA HERRAMIENTA DIAGNÓSTICA.** Proyecto educativo y experimental,
> no validado clínicamente.

112.120 imágenes de 30.805 pacientes, con 3-4 radiografías por paciente. Eso
hace que el **split por paciente** sea una necesidad real: dividir por imagen
metería al mismo paciente en train y test y mediría memorización.

**Referencia:** CheXNet (Rajpurkar et al. 2017) reporta AUROC **0,768** para
Pneumonia en este dataset. La etiqueta viene de NLP sobre informes, es ruidosa,
y la prevalencia es ~1,3%. El listón realista es ~0,75, no 0,85.

Código: https://github.com/GGGuardin/chest-xray-pneumonia

In [ ]:
import subprocess, sys, os, time
T0 = time.time()

subprocess.run(['rm', '-rf', '/tmp/repo'], check=False)
subprocess.run(['git', 'clone', '--depth', '1', '-q',
                'https://github.com/GGGuardin/chest-xray-pneumonia.git', '/tmp/repo'], check=True)
os.chdir('/tmp/repo')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'albumentations'], check=True)

import torch
print('torch', torch.__version__, '| GPU:',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO DISPONIBLE')
print('entradas montadas:', os.listdir('/kaggle/input'))

## 1. Manifiesto y split POR PACIENTE

El indexado de las 112k imágenes tarda unos minutos. Edad, sexo y proyección
AP/PA vienen en el CSV; son la base del análisis de subgrupos posterior.

In [ ]:
NIH = '/kaggle/input/data'
OUT = '/kaggle/working'
TARGET = 'Pneumonia'

!python -m src.prepare_data --dataset nih --root {NIH} --target {TARGET} \
    --out {OUT}/manifest_nih.csv

In [ ]:
import pandas as pd
from src.data import split_summary

df = pd.read_csv(f'{OUT}/manifest_nih.csv')
print(split_summary(df).to_string())
print('\nImagenes por paciente: media %.2f, maximo %d'
      % (len(df) / df.patient_id.nunique(), df.patient_id.value_counts().max()))
print('\nProyeccion:\n', df['view'].value_counts().to_string())
print('\nSexo:\n', df['sex'].value_counts().to_string())
print('\nEdad: mediana %.0f' % df.age.median())

assert (df.groupby('patient_id')['split'].nunique() == 1).all(), 'FUGA DE DATOS entre splits'
print('\nOK: ningun paciente aparece en mas de un split.')

## 2. Entrenamiento

DenseNet-121 preentrenada, 224x224, `pos_weight` (~75, por el desbalance
extremo), mixed precision. Checkpoint en cada mejora de AUROC de validación,
así que un corte de sesión no pierde el trabajo.

In [ ]:
!python -m src.train --config configs/nih.yaml \
    --manifest {OUT}/manifest_nih.csv \
    --out-dir {OUT}/runs/nih_densenet121

In [ ]:
import pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

os.makedirs(f'{OUT}/reports', exist_ok=True)
h = pd.read_csv(f'{OUT}/runs/nih_densenet121/history.csv')
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(h.epoch, h.train_loss, label='train'); ax[0].plot(h.epoch, h.val_loss, label='val')
ax[0].set_title('loss'); ax[0].set_xlabel('epoca'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(h.epoch, h.train_auroc, label='train'); ax[1].plot(h.epoch, h.val_auroc, label='val')
ax[1].axhline(0.768, ls='--', color='grey', label='CheXNet 0,768')
ax[1].set_title('AUROC'); ax[1].set_xlabel('epoca'); ax[1].legend(); ax[1].grid(alpha=.3)
fig.tight_layout(); fig.savefig(f'{OUT}/reports/curvas_entrenamiento.png', dpi=150)
print(h.to_string(index=False))

## 3. Evaluación en test interno (split por paciente)

In [ ]:
!python -m src.evaluate --checkpoint {OUT}/runs/nih_densenet121/best.pth \
    --manifest {OUT}/manifest_nih.csv --split test --out-dir {OUT}/reports/nih_test

## 4. Grad-CAM y auditoría de shortcut learning

Además de los mapas, se mide qué fracción de la energía del CAM cae en el marco
exterior: si es alta, el modelo mira bordes y marcadores, no el pulmón.

In [ ]:
!python -m src.explain --checkpoint {OUT}/runs/nih_densenet121/best.pth \
    --manifest {OUT}/manifest_nih.csv --split test --n 16 --out-dir {OUT}/reports/gradcam

## 5. Infradiagnóstico por subgrupos

FNR por sexo, grupo de edad, proyección AP/PA y sus intersecciones
(Seyyed-Kalantari et al., Nature Medicine 2021).

In [ ]:
!python -m src.fairness --predictions {OUT}/reports/nih_test/predictions.csv \
    --out-dir {OUT}/reports/fairness

## 6. Validación externa

El modelo, entrenado con adultos del NIH Clinical Center, se evalúa sobre
"Chest X-Ray Images (Pneumonia)": población **pediátrica**, otro país, otro
equipo y prevalencia del 74%. Es un cambio de dominio severo a propósito.
Criterio del proyecto: una caída de AUROC > 0,10 hay que documentarla y
analizarla, no esconderla.

In [ ]:
EXT = '/kaggle/input/chest-xray-pneumonia'
if os.path.exists(EXT):
    !python -m src.prepare_data --dataset kaggle_pneumonia --root {EXT} --out {OUT}/manifest_externo.csv
    !python -m src.evaluate --checkpoint {OUT}/runs/nih_densenet121/best.pth \
        --manifest {OUT}/manifest_externo.csv --split all --out-dir {OUT}/reports/externo_pediatrico
else:
    print('Dataset externo no montado; se omite la validacion externa.')

## 7. Resumen

In [ ]:
import json, glob, shutil

resumen = {'target': TARGET, 'referencia_chexnet_auroc': 0.768}
for nombre, ruta in [('test_interno_nih', f'{OUT}/reports/nih_test/metrics.json'),
                     ('externo_pediatrico', f'{OUT}/reports/externo_pediatrico/metrics.json')]:
    if os.path.exists(ruta):
        with open(ruta) as f:
            resumen[nombre] = json.load(f)

for ruta in [f'{OUT}/reports/fairness/fairness.json', f'{OUT}/reports/gradcam/shortcut_audit.json']:
    if os.path.exists(ruta):
        with open(ruta) as f:
            d = json.load(f)
        resumen[os.path.basename(os.path.dirname(ruta))] = {k: v for k, v in d.items() if k != 'detalle'}

interno = resumen.get('test_interno_nih', {}).get('auroc')
externo = resumen.get('externo_pediatrico', {}).get('auroc')
if interno and externo:
    resumen['caida_auroc_externa'] = round(interno - externo, 4)
resumen['minutos_totales'] = round((time.time() - T0) / 60, 1)

with open(f'{OUT}/resumen.json', 'w') as f:
    json.dump(resumen, f, indent=2, ensure_ascii=False)
print(json.dumps(resumen, indent=2, ensure_ascii=False))

# El manifiesto pesa varios MB y no aporta como salida del kernel
for f_ in glob.glob(f'{OUT}/manifest_*.csv'):
    shutil.move(f_, '/tmp/' + os.path.basename(f_))

print('\nArchivos de salida:')
for p in sorted(glob.glob(f'{OUT}/**/*', recursive=True)):
    if os.path.isfile(p):
        print(f'  {os.path.getsize(p)/1e6:8.2f} MB  {p}')